# Full experiment description (kidera factors)

## Importing and preparation

In [ ]:
import sys
import os
from pathlib import Path
from omegaconf import OmegaConf
import pandas as pd

project_root = Path().absolute().parent
sys.path.append(str(project_root))

### Path preparation

In [ ]:
%cd ..

In [ ]:
output_dir = 'full_exp_kidera'
weights_suf = 'weights'
report_suf = 'report'
embds_suf = 'embds'

### Constants

In [ ]:
embd_type = 'kidera'

## Import TCREnc scripts

In [ ]:
from tcrenc.tcrenc_run import main as tcrenc_run_main
from tcrenc.tcrenc_train import main as tcrenc_train_main
from tcrenc.tcrenc_validate import main as tcrenc_validate_main

## Kidera factors sequence representation experiment

Let's try to train autoencoder for kidera factors representation.
General pipeline:
```
1.configure model parameters ->  2. train model ->  3. Validate it on some test data -> 4. Make embeddings for future use
```

### 1.Model configuration

In [ ]:
with open('./tcrenc/configs/config_general.yaml') as f:
    general_config = OmegaConf.load(f)
print('General configurations:\n', general_config)

with open('./tcrenc/configs/config_kidera.yaml') as f:
    special_config = OmegaConf.load(f)
print('Special models configurations:\n', special_config)

* You can change parameters manually, or reassign some values here and save the dictionaries to the path specified below.

In [ ]:
# with open('./tcrenc/configs/config_general.yaml') as f:
#     general_config = OmegaConf.load(f)

# # Update
# general_config['LATENT_DIMS'] = 64

# with open('./tcrenc/configs/config_general.yaml', 'w') as f:
#     OmegaConf.save(general_config, f)

### 2. Model training 

#### Prepare training data

For epitope training we will use data from IEDB with following filters:
- linear peptides
- host: human

For cdr3 training we will use VDJdb.

#### Epitope model train

In [ ]:
script_name = 'tcrenc_train'

val_args = {
  "--input":'./dataset/epitopes_for_train/epitopes.csv',
  '--embed_type':embd_type,
  '--output':os.path.join(output_dir, weights_suf),
  '--split': '0.8'
}
logic_args = [
    "--weights_save"
    #'--cdr',
    #'--epitope'
]

argv_for_run = [script_name]
[argv_for_run.extend([k,v]) for k,v in val_args.items()]
if len(logic_args) != 0:
    argv_for_run.extend(logic_args)

sys.argv = argv_for_run

In [ ]:
tcrenc_train_main(sys.argv)

#### Update weights in config

In [ ]:
with open('./tcrenc/configs/config_kidera.yaml') as f:
    special_config = OmegaConf.load(f)

old_ep_weights = special_config['validate']['WEIGHTS_EPIOPE']
# Update
special_config['validate']['WEIGHTS_EPIOPE'] = os.path.join(output_dir, weights_suf, 'weights_kidera_antigen_epitope.pth')

with open('./tcrenc/configs/config_kidera.yaml', 'w') as f:
    OmegaConf.save(special_config, f)

#### CDR3 model train

In [ ]:
script_name = 'tcrenc_train'

val_args = {
  "--input":'VDJdb',
  '--embed_type':embd_type,
  '--output':os.path.join(output_dir, weights_suf),
  '--split': '0.8'
}
logic_args = [
    "--weights_save",
    '--cdr',
    #'--epitope'
]

argv_for_run = [script_name]
[argv_for_run.extend([k,v]) for k,v in val_args.items()]
if len(logic_args) != 0:
    argv_for_run.extend(logic_args)

sys.argv = argv_for_run

In [ ]:
tcrenc_train_main(sys.argv)

#### Update weights in config

In [ ]:
with open('./tcrenc/configs/config_kidera.yaml') as f:
    special_config = OmegaConf.load(f)

old_cdr_weights = special_config['validate']['WEIGHTS_CDR3']
# Update
special_config['validate']['WEIGHTS_CDR3'] = os.path.join(output_dir, weights_suf,'weights_kidera_cdr3.pth')

with open('./tcrenc/configs/config_kidera.yaml', 'w') as f:
    OmegaConf.save(special_config, f)

### 3. Model validation 

In [ ]:
script_name = 'tcrenc_validate'

val_args = {
  "--input":'VDJdb',
  '--embed_type':embd_type,
  '--output':os.path.join(output_dir, report_suf),
}
logic_args = [
    # "--weights_save",
    # '--cdr',
    # '--epitope'
]

argv_for_run = [script_name]
[argv_for_run.extend([k,v]) for k,v in val_args.items()]
if len(logic_args) != 0:
    argv_for_run.extend(logic_args)

sys.argv = argv_for_run

In [ ]:
tcrenc_validate_main(sys.argv)

### 4. Make embeddings with trained model

#### Update config

In [ ]:
with open('./tcrenc/configs/config_kidera.yaml') as f:
    special_config = OmegaConf.load(f)

# Update
special_config['run']['WEIGHTS_EPIOPE'] = os.path.join(output_dir, weights_suf, 'weights_kidera_antigen_epitope.pth')
special_config['run']['WEIGHTS_CDR3'] = os.path.join(output_dir, weights_suf,'weights_kidera_cdr3.pth')

with open('./tcrenc/configs/config_kidera.yaml', 'w') as f:
    OmegaConf.save(special_config, f)

#### Make embeddings

In [ ]:
script_name = 'tcrenc_run'

val_args = {
  "--input": 'VDJdb',
  '--embed_type':embd_type,
  '--output':os.path.join(output_dir, embds_suf),
}
logic_args = [
    # '--decoder',
    # '--cdr',
    # '--epitope'
]

argv_for_run = [script_name]
[argv_for_run.extend([k,v]) for k,v in val_args.items()]
if len(logic_args) != 0:
    argv_for_run.extend(logic_args)

sys.argv = argv_for_run

In [ ]:
tcrenc_run_main(sys.argv)

#### Use new embeddings to reconstruct sequences (Use our model as decoder)

In [ ]:
df = pd.read_csv(os.path.join(output_dir, embds_suf, 'embeddings_cdr3_kidera.csv'))
df.drop(columns='cdr3', inplace=True)
df.to_csv(os.path.join(output_dir, embds_suf, 'embeddings_cdr3_kidera_for_decoder.csv'), index=False)

In [ ]:
script_name = 'tcrenc_run'

val_args = {
  "--input": os.path.join(output_dir, embds_suf, 'embeddings_cdr3_kidera_for_decoder.csv'),
  '--embed_type':embd_type,
  '--output':os.path.join(output_dir, embds_suf),
}
logic_args = [
    '--decoder',
    '--cdr',
    # '--epitope'
]

argv_for_run = [script_name]
[argv_for_run.extend([k,v]) for k,v in val_args.items()]
if len(logic_args) != 0:
    argv_for_run.extend(logic_args)

sys.argv = argv_for_run

In [ ]:
tcrenc_run_main(sys.argv)

#### Re-update weights

In [ ]:
with open('./tcrenc/configs/config_kidera.yaml') as f:
    special_config = OmegaConf.load(f)

# Update
special_config['run']['WEIGHTS_EPIOPE'] = old_ep_weights
special_config['validate']['WEIGHTS_EPIOPE'] = old_ep_weights
special_config['run']['WEIGHTS_CDR3'] = old_cdr_weights
special_config['validate']['WEIGHTS_CDR3'] = old_cdr_weights 

with open('./tcrenc/configs/config_kidera.yaml', 'w') as f:
    OmegaConf.save(special_config, f)